In [1]:
from non_rigid.nets.pn2 import PN2Dense
from non_rigid.datasets.dedo import DedoDataset, DedoDataModule
import numpy as np
import torch
from omegaconf import OmegaConf
import json
import os
from pathlib import Path

import torch_geometric.data as tgd
import torch_geometric.loader as tgl

import rpad.visualize_3d.plots as vpl
from plotly import graph_objects as go
from plotly.subplots import make_subplots

# ignore TypedStorage warnings
import warnings
warnings.filterwarnings("ignore", message="TypedStorage is deprecated", category=UserWarning)

In [2]:
# torch settings
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Since most of us are training on 3090s+, we can use mixed precision.
torch.set_float32_matmul_precision("medium")
torch.manual_seed(42)

In [3]:
# set dataset config and create datamodule
dataset_cfg = OmegaConf.load("../configs/dataset/dedo.yaml")

overrides = {
    "train_size": 400,
    "scene": False,
    "world_frame": False,
    "scene_anchor": False,
    "rel_pose": False,
    "rel_pose_type": "translation",
    "center_type": "anchor_center",
    "action_context_center_type": "center",
    "cloth_geometry": "multi",
    "cloth_pose": "random",
    "sample_size_anchor": 512,
}

dataset_cfg = OmegaConf.merge(dataset_cfg, overrides)

# print(
#     json.dumps(
#         OmegaConf.to_container(dataset_cfg, resolve=True, throw_on_missing=False),
#         sort_keys=True,
#         indent=4,
#     )
# )

datamodule = DedoDataModule(
    batch_size=16,
    val_batch_size=4,
    num_workers=1,
    dataset_cfg=dataset_cfg,
)
datamodule.setup()



class PygDataset(tgd.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    
    def __len__(self):
        return len(self.dataset)
    
    def get(self, index):
        """
        Mini-wrapper for DedoDataset to return torch_geometric.data.Data. 
        "pos" is just the anchor point cloud, and "y" is the mean of the goal action point cloud in the anchor frame.
        """
        item = self.dataset[index]
        num_anchor_points = item["pc_anchor"].shape[0]

        data = tgd.Data(
            pos=item["pc_anchor"],
            y=item["pc"].mean(dim=0, keepdim=True).repeat(num_anchor_points, 1),
        )
        return data
    
    def __getitem__(self, index):
        return self.get(index)


train_dataset = PygDataset(datamodule.train_dataset)
val_dataset = PygDataset(datamodule.val_dataset)

train_batch_size = 16
val_batch_size = 4

train_loader = tgl.DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, num_workers=16)
val_loader = tgl.DataLoader(val_dataset, batch_size=val_batch_size, shuffle=False, num_workers=16)

In [4]:
# creating model

class FramePredictorSimple(torch.nn.Module):
    def __init__(self, out_channels):
        super(FramePredictorSimple, self).__init__()
        self.pn2 = PN2Dense(in_channels=0, out_channels=out_channels)

    def forward(self, batch):
        # make sure all of the point clouds in the batch have the same number of points
        ptr_diffs = torch.unique(batch.ptr[1:] - batch.ptr[:-1])
        if len(ptr_diffs) > 1:
            raise ValueError("All point clouds in the batch must have the same number of points.")
        else:
            num_points = ptr_diffs.item()

        output = self.pn2(batch)
        logits = output[:, [0]]
        means = output[:, 1:4]
        vars = output[:, 4:]

        # run vars through softplus to ensure positive values
        vars = torch.nn.functional.softplus(vars)

        # add mean residuals to points to get mean predictions
        # means = means + batch.pos

        # converting logits to probabilities
        probs = torch.nn.functional.softmax(logits.reshape(-1, num_points), dim=1).reshape(-1, 1)

        return {
            "probs": probs,
            "means": means,
            "vars": vars,
        }


In [5]:
# defining GMM loss function
class GMMLoss(torch.nn.Module):
    def __init__(self, eps=1e-6):
        """
        eps: value used to clamp var, for stability.
        """
        super(GMMLoss, self).__init__()
        self.eps = eps
    
    def forward(self, batch, pred):
        """
        batch: torch_geometric.data.Batch object. batch.y is (N, 3) tensor of target means.
        pred: dict with keys "probs", "means", "vars".
        """
        # make sure all of the point clouds in the batch have the same number of points
        ptr_diffs = torch.unique(batch.ptr[1:] - batch.ptr[:-1])
        if len(ptr_diffs) > 1:
            raise ValueError("All point clouds in the batch must have the same number of points.")
        else:
            num_points = ptr_diffs.item()

        targets = batch.y
        probs = pred["probs"]
        means = pred["means"]
        vars = pred["vars"]

        # clamp vars for stability
        # vars = torch.clamp(vars, min=self.eps)

        # multivariate homoscedastic gaussian likelihood
        norm_const = 1 / torch.sqrt(2 * np.pi**3 * vars)
        diff = targets - means
        likelihood = torch.exp(-0.5 * torch.sum(diff ** 2 / vars, dim=1, keepdim=True))
        likelihood = likelihood * norm_const
        
        # weighted sum of likelihoods across point cloud
        likelihood = likelihood * probs

        # log of sum of likelihoods per point cloud
        likelihood = likelihood.reshape(-1, num_points).sum(dim=1)
        loss = -torch.log(likelihood).mean()
        return loss

loss_fn = GMMLoss()

In [7]:
# create model - currently 5 for heteroscedastic variance
model = FramePredictorSimple(5)

# set up optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# training params
num_epochs = 50
val_every = 10

device = "cuda:0"
model.to(device)

total_losses = []
total_val_losses = []

min_val_loss = float("inf")

# basic training loop
for epoch in range(num_epochs):
    # train step
    model.train()
    epoch_loss = []
    for i, batch in enumerate(train_loader):
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        #print(i, pred.shape, batch.ptr)

        loss = loss_fn(batch, pred)
        loss.backward()
        optimizer.step()

        epoch_loss.append(loss.item())
    epoch_loss = np.mean(epoch_loss)
    total_losses.append(epoch_loss)

    # val step
    if (epoch + 1) % val_every == 0:
        model.eval()
        val_loss = []
        with torch.no_grad():
            for i, batch in enumerate(val_loader):
                batch = batch.to(device)
                pred = model(batch)

                loss = loss_fn(batch, pred)
                val_loss.append(loss.item())
        val_loss = np.mean(val_loss)
        total_val_losses.append(val_loss)

    # logging
    if (epoch + 1) % val_every == 0:
        print(f"Epoch {epoch}, Train Loss: {epoch_loss}, Val Loss: {val_loss}")

        # visualize some predictions if val loss is low
        val_fig = make_subplots(rows=2, cols=3, 
                                specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "scatter3d"}], 
                                       [{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "scatter3d"}]],
        )

        # visualize train
        for i in range(3):
            vis_batch = tgd.Batch.from_data_list([train_dataset[i]]).to(device)
            with torch.no_grad():
                pred = model(vis_batch)
            target = vis_batch.y.cpu().numpy()[0]
            pc_anchor = vis_batch.pos.cpu().numpy()
            means = pred["means"].cpu().numpy()
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=2, color="blue"),
                    x=pc_anchor[:, 0],
                    y=pc_anchor[:, 1],
                    z=pc_anchor[:, 2],
                ), row=1, col=i + 1
            )
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=2, color="red"),
                    x=means[:, 0],
                    y=means[:, 1],
                    z=means[:, 2],
                ), row=1, col=i + 1
            )
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=5, color="green"),
                    x=[target[0]],
                    y=[target[1]],
                    z=[target[2]],
                ), row=1, col=i + 1
            )
            # val_fig.add_traces(viz_traces(vis_batch, pred), rows=1, cols=i + 1)
        
        # visualize val
        for i in range(3):
            vis_batch = tgd.Batch.from_data_list([val_dataset[i]]).to(device)
            with torch.no_grad():
                pred = model(vis_batch)
            pc_anchor = vis_batch.pos.cpu().numpy()
            means = pred["means"].cpu().numpy()
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=2, color="blue"),
                    x=pc_anchor[:, 0],
                    y=pc_anchor[:, 1],
                    z=pc_anchor[:, 2],
                ), row=2, col=i + 1
            )
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=2, color="red"),
                    x=means[:, 0],
                    y=means[:, 1],
                    z=means[:, 2],
                ), row=2, col=i + 1
            )
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=5, color="green"),
                    x=[target[0]],
                    y=[target[1]],
                    z=[target[2]],
                ), row=2, col=i + 1
            )
            # val_fig.add_traces(viz_traces(vis_batch, pred), rows=2, cols=i + 1)

        # visualize predictions
        val_fig.update_layout(title_text=f"Epoch {epoch}, Train Loss: {epoch_loss} Val Loss: {val_loss}")
        val_fig.show(renderer="browser")
    else:
        print(f"Epoch {epoch}, Train Loss: {epoch_loss}")


# After training, plot the losses
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(1, len(total_losses) + 1), y=total_losses, name="Train Loss"))
fig.add_trace(go.Scatter(x=np.arange(val_every, (len(total_val_losses) + 1) * val_every, val_every), y=total_val_losses, name="Val Loss"))
fig.update_layout(title="Losses", xaxis_title="Epoch", yaxis_title="Loss")
fig.show()


Epoch 0, Train Loss: 7.194488468170166
Epoch 1, Train Loss: 3.457124376296997
Epoch 2, Train Loss: 3.1913834190368653
Epoch 3, Train Loss: 2.997151927947998
Epoch 4, Train Loss: 2.9726381874084473
Epoch 5, Train Loss: 3.020650501251221
Epoch 6, Train Loss: 2.9960244274139405
Epoch 7, Train Loss: 8.650344038009644
Epoch 8, Train Loss: 13.815510749816895
Epoch 9, Train Loss: 13.815510749816895, Val Loss: 13.815510749816895
Epoch 10, Train Loss: 13.815510749816895
Epoch 11, Train Loss: 13.815510749816895
Epoch 12, Train Loss: 13.815510749816895
Epoch 13, Train Loss: 13.815510749816895
Epoch 14, Train Loss: 13.815510749816895
Epoch 15, Train Loss: 13.815510749816895
Epoch 16, Train Loss: 13.815510749816895
Epoch 17, Train Loss: 13.815510749816895
Epoch 18, Train Loss: 13.815510749816895
Epoch 19, Train Loss: 13.815510749816895, Val Loss: 13.815510749816895
Epoch 20, Train Loss: 13.815510749816895
Epoch 21, Train Loss: 13.815510749816895
Epoch 22, Train Loss: 13.815510749816895
Epoch 23, Tr

In [8]:
# visualization code
model.eval()


for i in range(4):
    batch = tgd.Batch.from_data_list([val_dataset[i]]).to(device)


    with torch.no_grad():
        output = model(batch)

    # logits = output["logits"].cpu().numpy()
    means = output["means"].cpu().numpy()
    vars = output["vars"].cpu().numpy()
    # vars = np.linalg.norm(vars, axis=1)

    pc_anchor = batch.pos.cpu().numpy()

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker={
                "size": 5,
                "color": "blue",
            },
            x=pc_anchor[:, 0],
            y=pc_anchor[:, 1],
            z=pc_anchor[:, 2],
        )
    )
    fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker={
                "size": 5,
                "color": "green",
            },
            x=means[:, 0],
            y=means[:, 1],
            z=means[:, 2],
        )
    )
    traces = vpl._flow_traces(
        start=pc_anchor,
        flows=means - pc_anchor,
        flowscale=1.0,
        flowcolor="red",
    )
    fig.add_traces(traces[0])
    fig.show(renderer="browser")

In [9]:
means

array([[-0.0178299 ,  0.02845711,  7.927077  ],
       [-0.0178299 ,  0.02845711,  7.927077  ],
       [-0.0178299 ,  0.02845711,  7.927077  ],
       ...,
       [-0.0178299 ,  0.02845711,  7.927077  ],
       [-0.0178299 ,  0.02845711,  7.927077  ],
       [-0.0178299 ,  0.02845711,  7.927077  ]], dtype=float32)

In [ ]:
# set up model
class FramePredictorSimple(torch.nn.Module):
    def __init__(self, out_channels):
        super().__init__()

        self.pn = PN2Dense(
            in_channels = 0,
            out_channels = out_channels,
        )

    def forward(self, x):
        return self.pn(x)

model = FramePredictorSimple(7)